### Prerequisites:

Before we dive into the implementation, we have to install some prerequisites:

 - we use [pandas](https://pandas.pydata.org/docs/) to easily load and format our data
 - we use Mistral's open-source LLM [Mistral 7B-Instruct-v0.3](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3) for zero-shot classification

In [1]:
%%capture
# install dependencies:
!pip install numpy matplotlib seaborn torch pandas scikit-learn transformers accelerate tqdm datasets

### Data processing

In [2]:
from datasets import load_dataset

train_data = load_dataset("Karim-Gamal/SemEval-2018-Task-2-english-emojis", split="train")
test_data  = load_dataset("Karim-Gamal/SemEval-2018-Task-2-english-emojis", split="test")

print(f"Train set size: {len(train_data)} examples")
print(f"Test set size:  {len(test_data)} examples")

Train set size: 454020 examples
Test set size:  50000 examples


In [3]:
# SemEval-2018 Task 2 English emoji labels (index → emoji):
index2label = {
    0: "❤",  1: "😍", 2: "😂",  3: "💕", 4: "🔥",
    5: "😊",  6: "😎", 7: "✨",  8: "💙", 9: "😘",
    10: "📷", 11: "🇺🇸", 12: "☀", 13: "💜", 14: "😉",
    15: "💯", 16: "😁", 17: "🎄", 18: "📸", 19: "😜"
}
index2label

{0: '❤',
 1: '😍',
 2: '😂',
 3: '💕',
 4: '🔥',
 5: '😊',
 6: '😎',
 7: '✨',
 8: '💙',
 9: '😘',
 10: '📷',
 11: '🇺🇸',
 12: '☀',
 13: '💜',
 14: '😉',
 15: '💯',
 16: '😁',
 17: '🎄',
 18: '📸',
 19: '😜'}

In [4]:
# convert datasets to dataframes:
train_df = train_data.to_pandas()
test_df  = test_data.to_pandas()

# filter empty sentences:
train_df = train_df[train_df['sentence'].str.strip().astype(bool)]
test_df  = test_df[test_df['sentence'].str.strip().astype(bool)]

train_df.sample()

,sentence,label
89472,I don't think I could be more blessed ️ luckie...,0


In [5]:
from sklearn.model_selection import train_test_split

# sample 2k training examples stratified by label:
train_df, _ = train_test_split(
    train_df, stratify=train_df['label'], train_size=2000, random_state=42
)

# sample 1k test examples stratified by label:
test_df, _ = train_test_split(
    test_df, stratify=test_df['label'], train_size=1000, random_state=42
)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")

Train: 2000, Test: 1000


In [6]:
# map numeric labels to string labels:
train_df['label'] = train_df['label'].map(index2label)
test_df['label']  = test_df['label'].map(index2label)

In [7]:
import numpy as np

# select input and label from data:
X_train = train_df['sentence'].to_numpy()
y_train = train_df['label'].to_numpy()

X_test  = test_df['sentence'].to_numpy()
y_test  = test_df['label'].to_numpy()

classes = sorted(set(y_train))
print("Classes:", len(classes))

Classes: 20


### Instantiate Mistral 7B

In order to be able to use huggingface's Mistral 7B-Instruct-v0.3 models, we first need to log in to huggingface (you can request access to the model [here](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)).

In [8]:
# import getpass
# from huggingface_hub import login
# login(getpass.getpass('Enter your huggingface API-key:'))

In this tutorial we will use the standard huggingface text-generation pipeline (compressed to 16 bit floating point weights) for the instruction tuned Mistral 7B-Instruct-v0.3 model.

In [9]:
import transformers
import torch

# create llm pipeline:
llm = transformers.pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.3",
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto"
)

# Get special tokens for later:
bos_token_id = llm.tokenizer.bos_token_id
eos_token_id = llm.tokenizer.eos_token_id
pad_token_id = llm.tokenizer.pad_token_id or llm.tokenizer.eos_token_id

`torch_dtype` is deprecated! Use `dtype` instead!


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

### Results Handling Helper

In [10]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import json

def save_results(y_true, y_pred, classes, filename="results.json"):
    report   = classification_report(y_true, y_pred, labels=classes, zero_division=0, output_dict=True)
    accuracy = accuracy_score(y_true, y_pred)
    cm       = confusion_matrix(y_true, y_pred, labels=classes)
    results  = {"classification_report": report, "accuracy": accuracy,
                "confusion_matrix": cm.tolist(), "labels": classes}
    with open(filename, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to: '{filename}'")

In [11]:
import json
import numpy as np

def save_prompt_lengths(prompt_lengths, input_lengths, shots_lengths, num_shots, filename="prompt_lengths.json"):
    def stats(v): return {"mean": float(np.mean(v)), "min": int(np.min(v)),
                          "max": int(np.max(v)), "std": float(np.std(v)), "values": v}
    data = {"prompt_lengths": stats(prompt_lengths), "input_lengths": stats(input_lengths),
            "shots_lengths": stats(shots_lengths), "num_shots": stats(num_shots)}
    with open(filename, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Prompt length statistics saved to: '{filename}'")

### Zero-shot Classification

Build a prompt listing **all possible class labels** explicitly. No labelled examples are provided — the model must rely solely on the class names and task description.

In [12]:
def get_zero_shot_prompt(text, classes):
    """Build a zero-shot classification prompt listing all possible classes."""
    replace_qm = lambda s: s.replace('"', "'")
    class_list = ", ".join(classes)
    prompt = (
        "We classify tweets by their most likely emoji. "
        f"The possible classes are: {class_list}.\n\n"
        "Please predict the correct class for the following sample. "
        "Output only the class label, nothing else."
    )
    return {"role": "user", "content": f'{prompt}\n\n\"{replace_qm(text)}\" => '}


# print a sample prompt to verify:
print(get_zero_shot_prompt(X_test[0], classes)["content"])

We classify tweets by their most likely emoji. The possible classes are: ☀, ✨, ❤, 🇺🇸, 🎄, 💕, 💙, 💜, 💯, 📷, 📸, 🔥, 😁, 😂, 😉, 😊, 😍, 😎, 😘, 😜.

Please predict the correct class for the following sample. Output only the class label, nothing else.

"Happy 4th of July weekend #babe # mu'rica @ Flower Mound, Texas" => 


In [13]:
from tqdm import tqdm
import numpy as np

def zero_shot_batch(test, classes):
    predictions    = []
    prompt_lengths = []
    input_lengths  = []
    shots_lengths  = []  # always 0 — no shots
    num_shots      = []  # always 0

    for t in tqdm(test, desc="Zero-shot inference"):
        prompt = get_zero_shot_prompt(t, classes)

        prompt_lengths.append(len(llm.tokenizer(prompt["content"])["input_ids"]))
        input_lengths.append(len(llm.tokenizer(t)["input_ids"]))
        shots_lengths.append(0)
        num_shots.append(0)

        output = llm(
            [prompt],
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            max_new_tokens=5,
            do_sample=False,
            temperature=None,
            top_p=None
        )
        predictions.append(output[0]["generated_text"][-1]["content"].strip())

    return predictions, prompt_lengths, input_lengths, shots_lengths, num_shots


predictions, prompt_lengths, input_lengths, shots_lengths, num_shots = zero_shot_batch(X_test, classes)

Zero-shot inference:   0%|          | 0/1000 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'eos_token_id', 'top_p', 'bos_token_id', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zero-shot inference:   0%|          | 1/1000 [00:00<13:26,  1.24it/s]

Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zero-shot inference: 100%|█████████▉| 999/1000 [05:09<00:00,  3.13it/s]

Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zero-shot inference: 100%|██████████| 1000/1000 [05:10<00:00,  3.02it/s]

Zero-shot inference: 100%|██████████| 1000/1000 [05:10<00:00,  3.22it/s]

In [14]:
predictions[:10]

['🇺🇸', '☀', '🏠', '🎄', '🇺🇸', '📷', '😊 (Smiling', '🇺🇸', '😊', '📷']

In [15]:
from sklearn.metrics import classification_report, accuracy_score

print("Classification Report:\n", classification_report(y_test, predictions, labels=list(classes), zero_division=0))
print("Accuracy:", accuracy_score(y_test, predictions))

Classification Report:
               precision    recall  f1-score   support

           ☀       0.25      0.44      0.32        25
           ✨       0.30      0.22      0.25        55
           ❤       0.27      0.01      0.03       216
          🇺🇸       0.24      0.79      0.37        39
           🎄       0.20      0.90      0.33        31
           💕       0.33      0.04      0.07        52
           💙       0.03      0.03      0.03        31
           💜       0.00      0.00      0.00        22
           💯       0.00      0.00      0.00        25
           📷       0.07      0.28      0.12        29
           📸       0.00      0.00      0.00        48
           🔥       0.78      0.09      0.17        74
           😁       0.00      0.00      0.00        23
           😂       0.45      0.11      0.18        91
           😉       0.00      0.00      0.00        26
           😊       0.05      0.16      0.07        32
           😍       0.12      0.02      0.04        97
   

In [16]:
save_results(y_test, predictions, list(classes), filename="/home/v25/ippa6201/cicle-evaluation/semeval-18/results/predictions/semeval-18-mistral-7b-v0.3-zeroshot-2.0k-samples.json")

Results saved to: '/home/v25/ippa6201/cicle-evaluation/semeval-18/results/predictions/semeval-18-mistral-7b-v0.3-zeroshot-2.0k-samples.json'


In [17]:
save_prompt_lengths(prompt_lengths, input_lengths, shots_lengths, num_shots, filename="/home/v25/ippa6201/cicle-evaluation/semeval-18/results/lengths/semeval-18-mistral-7b-v0.3-zeroshot-2.0k-samples.json")

Prompt length statistics saved to: '/home/v25/ippa6201/cicle-evaluation/semeval-18/results/lengths/semeval-18-mistral-7b-v0.3-zeroshot-2.0k-samples.json'
